# 04 — Deterministic baselines and the evaluation harness

**Estimated time:** 45 minutes<br>
**Prerequisites:** 03 — Portable records and leakage-safe splits<br>
**Learner-produced evidence:** validation metrics for majority and locked keyword/rule baselines

## Learning objectives

- Contrast a sanity-floor majority baseline with a meaningful rule baseline.
- Interpret macro and weighted classification metrics.
- Score structured output, response policy, performance, slices, and errors separately.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

A fine-tuned model is not useful merely because it produces non-random answers. It must beat simple approaches that are cheaper, faster, easier to inspect, and often surprisingly strong. Deterministic baselines also exercise the exact output and evaluation contracts before model inference adds cost or uncertainty.

## Key terms in plain language

- **prediction:** the answer emitted by a method for one input.
- **majority baseline:** a sanity-floor classifier that always predicts the most common training label.
- **deterministic baseline:** a repeatable method such as rules or retrieval whose output is fixed for the same input.
- **precision:** among items predicted as a class, the fraction whose label is actually that class.
- **recall:** among items that truly belong to a class, the fraction the method finds.
- **F1:** the harmonic mean of precision and recall; it is high only when both are reasonably high.
- **macro F1:** the unweighted mean of per-class F1, giving each class equal influence.
- **weighted F1:** the mean of per-class F1 weighted by class frequency, so common classes influence it more.
- **schema validity:** whether the output parses and satisfies all required types, fields, and allowed values.


## Mental model — how to think about this

Build a minimum-bar ladder. The majority baseline is the floor: it catches broken metrics and class imbalance. A task-aware deterministic baseline is the meaningful rung: it represents what an inexpensive inspectable system can already do. Prompted and fine-tuned models must be compared against the strongest relevant rung with the same records and evaluator.

### Running example

A majority baseline may call every message the most frequent intent and miss `recover_password`. A keyword rule may notice `forgot` plus `password` and emit the right typed object. The future model must justify why its extra complexity is better than that task-aware rule, not merely better than majority guessing.

### Questions to ask before continuing

- What simple method could solve a large fraction of this task without a generative model?
- Which error is more costly for each class: a false positive or a false negative?
- Would an output with the right label but invalid JSON be usable by the downstream system?
- Do global metrics conceal a rare but important intent or policy failure?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Fit or configure baselines from training data only.** Validation measures choices; it is not a source for silently extending rules after seeing results.
- **Use the same evaluation harness for every method.** Identical IDs, labels, parsers, policy checks, and timing boundaries make comparisons meaningful.
- **Report complementary metrics.** Pair accuracy with macro and weighted F1, per-label counts, schema validity, response-policy compliance, and bounded errors.
- **Keep unsupported inputs explicit.** A known abstain or unsupported intent is preferable to inventing a confident in-scope answer.
- **Retain the simple winner.** If a rule system meets the contract, a model must justify added latency, cost, nondeterminism, and governance burden—not merely tie its score.

## Common mistakes and why they fail

- **Reporting only accuracy.** A majority class can dominate it while minority classes fail completely.
- **Counting invalid JSON as only a wrong class.** Format validity is an independent operational contract.
- **Choosing an intentionally weak baseline.** Beating a straw man provides little evidence for adoption.
- **Editing rules after each validation result without versioning.** That is still model selection and must be recorded as a new change before final test evaluation.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [scikit-learn metrics and scoring guide](https://scikit-learn.org/stable/modules/model_evaluation.html)
- **Tool guidance:** [scikit-learn dummy estimators as baseline values](https://scikit-learn.org/stable/modules/model_evaluation.html#dummy-estimators)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Fit only on training evidence

The majority baseline learns its fixed label from train. The keyword/rule
baseline learns label counts, token weights, category mappings, and
escalation defaults from train **and** includes human-authored phrase and
escalation rules locked in source code before validation. It is therefore
transparent and input-aware, but not wholly train-derived. Validation
estimates behavior while methods may still change; test remains unopened.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import (
    DeterministicInferenceConfig,
    KeywordRuleBaseline,
    MajorityBaseline,
    evaluate_predictions,
    format_error_analysis,
    recheck_evaluation_session,
    start_evaluation_session,
)
from aai_local_finetuning.learning import (
    load_support_splits,
    report_row,
    support_contract,
)

splits = load_support_splits(include_test=False)
allowed_intents, _ = support_contract(splits.train)
baseline_evaluation_session = start_evaluation_session()
majority = MajorityBaseline.fit(splits.train)
keyword = KeywordRuleBaseline.fit(splits.train)

## Score through one strict harness

The same evaluator parses JSON, validates the schema, rejects unsupported
labels, checks response policy, calculates classification metrics, and
summarizes latency, output tokens, memory, slices, and bounded errors.

Read the evaluation ladder from basic usability toward task quality:

`parseable JSON → schema-valid fields → allowed label → correct task answer →`
`course-policy-compliant response → acceptable latency/tokens/memory`

One record can fail several layers, so error-kind counts can overlap and
must not be added together to infer a number of failed records.

An **evaluation session** is the integrity boundary around this work. It
records the governed source and exact installed packages before fitting or
inference, supplies that same contract to scoring, and is rechecked after
the report exists. This makes the report describe the code that actually
created the fitted predictor state, generated predictions, and scored them,
including a change that was later restored.

Each report also carries an **inference configuration**. These two
baselines declare `mode="deterministic"`, which is an important honest
claim: they used no checkpoint, prompt recipe, sampler, or token budget.
Model-free evidence should not become stale merely because an unrelated
model file changes.


In [ ]:
baseline_reports = {
    "majority": evaluate_predictions(
        splits.validation,
        majority.predict_many(splits.validation),
        supported_intents=allowed_intents,
        evaluation_session=baseline_evaluation_session,
        inference_config=DeterministicInferenceConfig(method="majority"),
    ),
    "keyword-rule": evaluate_predictions(
        splits.validation,
        keyword.predict_many(splits.validation),
        supported_intents=allowed_intents,
        evaluation_session=baseline_evaluation_session,
        inference_config=DeterministicInferenceConfig(method="keyword-rule"),
    ),
}
recheck_evaluation_session(baseline_evaluation_session)
pd.DataFrame([report_row(name, report) for name, report in baseline_reports.items()])

## How to read the score table

| Field | Better direction | Question it answers |
|---|---:|---|
| Intent accuracy | Higher | How often was the exact intent correct? |
| Macro precision / recall / F1 | Higher | How well does each intent perform when every intent has equal influence? |
| Weighted F1 | Higher | How well does the observed label mix perform, giving common intents more influence? |
| Category / escalation accuracy | Higher | Were the other authoritative target fields correct? |
| JSON parse / schema validity | Higher | Can code read the output, and does it satisfy the exact contract? |
| Unsupported-intent rate | Lower | How often did the method invent a label outside the vocabulary? |
| Response-policy compliance | Higher | Did wording pass this course's narrow lexical policy? |
| Latency, tokens, memory | Context or lower | What local resource cost accompanied the result? |

“Support” in a per-intent table means the number of true evaluation
examples for that intent; it does not mean customer-support quality.
Passing the response policy does not establish truthfulness, helpfulness,
privacy, or general safety.


## Why macro F1 matters

Accuracy can be dominated by frequent labels. Macro F1 gives each intent
equal weight; weighted F1 reflects observed support. Report both. The
majority method is a sanity floor and is not considered meaningful for
promotion, even when its JSON happens to be valid.


In [ ]:
pd.DataFrame(
    {
        "intent": list(baseline_reports["keyword-rule"].classification.per_intent_f1),
        "f1": list(
            baseline_reports["keyword-rule"].classification.per_intent_f1.values()
        ),
        "support": [
            baseline_reports["keyword-rule"].by_intent[intent].count
            for intent in baseline_reports["keyword-rule"].classification.per_intent_f1
        ],
    }
).sort_values("f1").head(10)

## Inspect what the transparent baseline learned

The displayed weighted terms come only from training records. Separate
phrase rules and escalation triggers are human-authored course rules.
Both are useful for debugging and reveal brittleness: lexical shortcuts
can fail on paraphrases, ambiguity, negation, or overlapping vocabulary.


In [ ]:
{intent: keyword.keywords_by_intent[intent][:6] for intent in list(allowed_intents)[:8]}

## Bounded error evidence

Error kinds remain separate: invalid JSON, schema mismatch, unsupported
intent, intent/category/escalation errors, and response-policy failures.
Only bounded masked previews are retained.


In [ ]:
print(format_error_analysis(baseline_reports["keyword-rule"]))

## Exercise — explain the meaningful baseline

Choose one learned keyword group and predict a likely failure mode.
Success means your explanation names the shortcut and a validation slice
or example type that could expose it.


In [ ]:
inspected_intent = allowed_intents[0]
likely_failure = (
    "A paraphrase with none of the learned high-weight terms may fall back "
    "to a different intent; inspect standard and hard validation examples."
)
{
    "intent": inspected_intent,
    "keywords": keyword.keywords_by_intent[inspected_intent][:6],
    "hypothesis": likely_failure,
}

**Hint:** transparent rules are valuable because a failure hypothesis can
be tied to visible features instead of model mythology.


## Checkpoint

Record which baseline is meaningful and why valid JSON alone is not a
strong result.

**Next:** `05_prompt_baselines.ipynb` holds weights fixed and changes only
the prompt evidence on validation data.
